> **🎯 Important**
>
**Durée estimée** : 10 à 12 heures  
**Prérequis** : TP Module 7 (chatbot RAG) + toutes les notions du Module 8  
**Objectif** : déployer **réellement en ligne** ton chatbot TechVolt avec API, Docker, monitoring et CI/CD


> **💡 Astuce**
>
## 🎓 Le projet qui boucle ton parcours
À la fin de ce TP, tu auras :

- ✅ Un **chatbot accessible par URL publique** (que tu peux partager à un recruteur)
- ✅ Un **repo GitHub propre** avec README pro
- ✅ Un **pipeline CI/CD automatique**
- ✅ Du **monitoring** structuré
- ✅ Un projet **complet, professionnel** sur ton CV

**C'est LE projet** qui te démarque des candidats "purement académiques" en entretien.


# 🎯 Contexte

C'est ta dernière mission pour TechVolt. L'équipe a adoré ton chatbot RAG. Mais il tourne en local sur ton PC. Le CTO te dit :

> *« Ton chatbot est génial, mais on ne va pas demander à chaque commercial de le lancer en local. Mets-le en ligne, on veut juste partager une URL au support. Et automatise le déploiement, hors de question d'attendre que tu sois dispo pour pousser une correction. Et bien sûr : monitoring obligatoire, on doit savoir si ça déconne. »*

## 🎁 Les objectifs business

1. **URL publique** accessible 24/7 (pas un localhost)
2. **Interface conversationnelle** (pas une API JSON brute)
3. **Pipeline CI/CD** : push GitHub → déploiement automatique
4. **Monitoring** : logs structurés, métriques de base
5. **Reproductibilité** : Docker, secrets sécurisés, doc complète
6. **Coût** : **0€** (Hugging Face Spaces gratuit + GitHub gratuit + Groq gratuit)

## 🗺️ Plan du TP

7 parties :

1. **Préparer le projet** (structure, code refactoré, dépendances)
2. **Interface Gradio** : transformer le code en app conversationnelle
3. **API FastAPI** : exposer aussi en mode programmatique
4. **Tests** : pytest + golden dataset
5. **Docker** : containeriser proprement
6. **Déploiement HF Spaces** : ton premier déploiement réel
7. **CI/CD GitHub Actions** : tout devient automatique

> **ℹ️ Note**
>
## 📦 Ce TP est un VRAI projet
Contrairement aux notions précédentes, ce TP **ne s'exécute pas** essentiellement dans un notebook. Tu vas :
- Créer un **dossier projet** sur ton ordi
- Configurer plusieurs comptes (HF, GitHub si pas fait)
- **Pousser du code** sur GitHub
- **Tester en ligne** ton déploiement

Garde un **terminal**, **VS Code** (ou ton IDE) et un **navigateur** ouverts.


# 🛠️ Mise en route

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import os
import json
from pathlib import Path

print("✅ Environnement prêt")

## 📦 Comptes et outils nécessaires

Avant de commencer, assure-toi d'avoir :

- [ ] **GitHub** : compte créé, Git installé localement
- [ ] **Hugging Face** : crée un compte sur [huggingface.co](https://huggingface.co)
- [ ] **Groq** : ta clé API (déjà obtenue au Module 7)
- [ ] **Docker Desktop** installé (Notion 8.3)
- [ ] **Python 3.10+** dans ton environnement local

---

# Partie 1 — Structure du projet

## 🏗️ Étape 1.1 — Créer l'arborescence

> **ℹ️ Note**
>
## 📝 À faire

Crée un nouveau dossier `techvolt-chatbot-prod/` avec cette structure exacte :

```
techvolt-chatbot-prod/
├── .github/
│   └── workflows/
│       └── ci.yml                  # CI/CD (Partie 7)
├── src/
│   ├── __init__.py
│   ├── chatbot.py                  # logique du chatbot (depuis TP M7)
│   ├── tools.py                    # outils métier
│   └── monitoring.py               # logging structuré
├── data/
│   └── knowledge_base.json         # corpus TechVolt (depuis TP M7)
├── tests/
│   ├── __init__.py
│   ├── test_api.py
│   └── test_chatbot.py
├── app.py                          # entrée principale (Gradio + FastAPI)
├── Dockerfile
├── requirements.txt
├── .gitignore
├── .env.example
├── .dockerignore
└── README.md
```

**Astuce** : sur Linux/Mac, tu peux créer tout ça en une commande :

```bash
mkdir -p techvolt-chatbot-prod/{.github/workflows,src,data,tests}
cd techvolt-chatbot-prod
touch app.py Dockerfile requirements.txt .gitignore .env.example .dockerignore README.md
touch src/__init__.py src/chatbot.py src/tools.py src/monitoring.py
touch tests/__init__.py tests/test_api.py tests/test_chatbot.py
touch .github/workflows/ci.yml
```


## 📋 Étape 1.2 — Récupérer le corpus du TP M7

> **ℹ️ Note**
>
## 📝 À faire

1. Copie le fichier **`knowledge_base.json`** (du TP M7) dans `data/knowledge_base.json`
2. Vérifie qu'il contient bien **26 documents**


> **💡 Astuce**
>
## ✅ Vérification

In [ ]:
import json
from pathlib import Path

corpus_path = Path("data/knowledge_base.json")
with open(corpus_path, "r", encoding="utf-8") as f:
    corpus = json.load(f)

print(f"Nombre de documents : {len(corpus)}")
assert len(corpus) == 26, "Corpus incomplet !"

## 🔐 Étape 1.3 — Fichier `.gitignore`

> **ℹ️ Note**
>
## 📝 À faire

Crée un `.gitignore` complet pour exclure les fichiers sensibles et les builds :

> 💡 **Tu bloques ?** Consulte la correction sur le site web du cours en dépliant le bloc « Voir la correction ».

## 🔑 Étape 1.4 — Fichier `.env.example`

> **ℹ️ Note**
>
## 📝 À faire

Crée un `.env.example` (à commiter, **sans** vraies valeurs) qui documente les variables nécessaires :

> 💡 **Tu bloques ?** Consulte la correction sur le site web du cours en dépliant le bloc « Voir la correction ».

## 📦 Étape 1.5 — `requirements.txt`

> **ℹ️ Note**
>
## 📝 À faire

Crée le `requirements.txt` avec les dépendances nécessaires (versions fixées) :

> 💡 **Tu bloques ?** Consulte la correction sur le site web du cours en dépliant le bloc « Voir la correction ».

---

# Partie 2 — Refactorer le code en modules

Le code du TP M7 est dans un notebook. Pour passer en production, il faut le **modulariser**.

## 🧩 Étape 2.1 — `src/tools.py`

> **ℹ️ Note**
>
## 📝 À faire

Extrais les **outils métier** du TP M7 dans `src/tools.py` :
- `diagnostic_erreur(code)`
- `creer_ticket_support(probleme, urgence)`
- `prendre_rdv_installation(date_iso, creneau)`
- `get_horaires_sav(jour)`

Ajoute aussi la **liste des schémas JSON** `OUTILS_SCHEMAS` et le **dict de routage** `OUTILS_FONCTIONS`.

> 💡 **Tu bloques ?** Consulte la correction sur le site web du cours en dépliant le bloc « Voir la correction ».

## 📊 Étape 2.2 — `src/monitoring.py`

> **ℹ️ Note**
>
## 📝 À faire

Crée un **logger structuré** réutilisable :

> 💡 **Tu bloques ?** Consulte la correction sur le site web du cours en dépliant le bloc « Voir la correction ».

## 🤖 Étape 2.3 — `src/chatbot.py`

> **ℹ️ Note**
>
## 📝 À faire

Extrais la **classe `ChatbotTechVolt`** du TP M7 dans `src/chatbot.py`.

Adapte-la pour :
1. Charger le corpus depuis `data/knowledge_base.json` au démarrage
2. Utiliser `src/monitoring.py` pour les logs
3. Avoir une méthode `chat(question: str) -> str` simple
4. Lire `GROQ_API_KEY` depuis l'environnement

> 💡 **Tu bloques ?** Consulte la correction sur le site web du cours en dépliant le bloc « Voir la correction ».

---

# Partie 3 — Interface Gradio (le visuel cool)

C'est l'interface utilisateur que les inconnus verront. **Gradio** permet de créer un chat en 20 lignes.

## 🎨 Étape 3.1 — `app.py`

> **ℹ️ Note**
>
## 📝 À faire

Crée le fichier **`app.py`** qui :
1. Charge le chatbot (avec le décor de chargement)
2. Crée une interface Gradio `ChatInterface`
3. Intègre quelques **exemples** de questions pour aider les utilisateurs
4. Lance Gradio sur le port 7860 (standard HF Spaces)

> 💡 **Tu bloques ?** Consulte la correction sur le site web du cours en dépliant le bloc « Voir la correction ».

## 🧪 Étape 3.2 — Test local de l'interface Gradio

> **ℹ️ Note**
>
## 📝 À faire

Dans ton terminal, depuis le dossier du projet :

```bash
# Activer ton venv si besoin
pip install -r requirements.txt

# Lancer
python app.py
```

Tu dois voir :
```
Running on local URL:  http://0.0.0.0:7860
```

Ouvre [http://localhost:7860](http://localhost:7860) → tu vois ton chatbot. **Teste les exemples** !


---

# Partie 4 — Tests automatisés

## 🧪 Étape 4.1 — Tests API (`tests/test_api.py`)

> **ℹ️ Note**
>
## 📝 À faire

Écris des **tests pytest** pour valider que ton chatbot fonctionne. Inclus :

1. Un test de **smoke** (le chatbot répond sans crasher)
2. Un test **golden dataset** sur 3 questions clés
3. Un test que les **outils** sont bien appelés

> 💡 **Tu bloques ?** Consulte la correction sur le site web du cours en dépliant le bloc « Voir la correction ».

---

# Partie 5 — Containerisation Docker

## 🐳 Étape 5.1 — `Dockerfile`

> **ℹ️ Note**
>
## 📝 À faire

Crée un Dockerfile **propre** :

- Image `python:3.12-slim`
- User non-root
- Cache pip optimisé
- Healthcheck sur l'endpoint Gradio (port 7860)
- Variables d'env explicites

> 💡 **Tu bloques ?** Consulte la correction sur le site web du cours en dépliant le bloc « Voir la correction ».

## 📁 Étape 5.2 — `.dockerignore`

> 💡 **Tu bloques ?** Consulte la correction sur le site web du cours en dépliant le bloc « Voir la correction ».

## 🚀 Étape 5.3 — Build et test local

> **ℹ️ Note**
>
## 📝 À faire

Dans ton terminal :

```bash
# Build
docker build -t techvolt-chatbot:dev .

# Run (avec ta clé API)
docker run --rm -p 7860:7860 \
  -e GROQ_API_KEY=$GROQ_API_KEY \
  techvolt-chatbot:dev
```

Ouvre [http://localhost:7860](http://localhost:7860) → tu vois ton chatbot tournant **dans Docker** ! 🐳


---

# Partie 6 — Déploiement sur Hugging Face Spaces

## 🤗 Étape 6.1 — Créer un Space

> **ℹ️ Note**
>
## 📝 À faire

1. Va sur [huggingface.co](https://huggingface.co), connecte-toi
2. Clique **+ New** → **Space**
3. Configure :
   - **Owner** : ton username
   - **Space name** : `techvolt-chatbot` (ou autre)
   - **License** : MIT (recommandé pour projets pro)
   - **SDK** : choisis **Gradio**
   - **Hardware** : CPU basic (gratuit)
   - **Visibility** : Public

4. Note l'URL : `https://huggingface.co/spaces/TONUSERNAME/techvolt-chatbot`


## 🔐 Étape 6.2 — Configurer les secrets

> **ℹ️ Note**
>
## 📝 À faire

Dans ton Space, **Settings** → **Variables and secrets** :

1. Ajoute un **secret** :
   - Name : `GROQ_API_KEY`
   - Value : ta vraie clé Groq

**IMPORTANT** : un **secret** est privé (chiffré). Une **variable** est publique (visible par tous). Pour les clés API → toujours **secret**.


## 📤 Étape 6.3 — Pousser le code

Tu as 2 options pour pousser ton code sur HF Spaces.

### Option A : pousser directement (rapide, manuel)

> **ℹ️ Note**
>
## 📝 À faire

```bash
# Cloner le Space (vide pour l'instant)
git clone https://huggingface.co/spaces/TONUSERNAME/techvolt-chatbot
cd techvolt-chatbot

# Copier ton code dedans
cp -r ../techvolt-chatbot-prod/* ../techvolt-chatbot-prod/.* .

# Push
git add .
git commit -m "Initial deployment"
git push
```

Va sur l'URL de ton Space. Tu vois "Building..." → attends ~5 minutes → ton chatbot est **EN LIGNE** ! 🎉


### Option B : 2 remotes (mieux pour la suite)

Pour avoir GitHub **ET** HF Spaces synchronisés :

```bash
# Dans ton dossier de projet
git init
git remote add origin https://github.com/USERNAME/techvolt-chatbot-prod.git
git remote add hf https://huggingface.co/spaces/USERNAME/techvolt-chatbot

git add .
git commit -m "Initial commit"

# Push sur GitHub
git push -u origin main

# Push sur HF Spaces
git push hf main
```

> **💡 Astuce**
>
## 💡 Authentification HF
Si Git te demande un mot de passe pour `huggingface.co`, utilise un **token** (pas ton mot de passe) :
1. [huggingface.co/settings/tokens](https://huggingface.co/settings/tokens)
2. **New token** avec rôle **Write**
3. Copie-le, colle-le quand demandé


## ✅ Étape 6.4 — Vérifier le déploiement

> **ℹ️ Note**
>
## 📝 À faire

1. Va sur l'URL de ton Space
2. Attends le **build** (Logs onglet "App")
3. Quand "Running" est affiché → teste avec une question
4. **Partage l'URL** à un·e ami·e !


> **⚠️ Attention**
>
## ⏱️ Cold start
HF Spaces gratuit met ton app **en pause** après 48h sans activité. Au prochain accès, il faut **~2-3 min** pour redémarrer (cold start). C'est normal pour le tier gratuit.


---

# Partie 7 — CI/CD avec GitHub Actions

Maintenant qu'on a un déploiement manuel, **automatisons** : à chaque `git push` sur GitHub, on déploie automatiquement sur HF Spaces.

## 🔄 Étape 7.1 — Workflow GitHub Actions

> **ℹ️ Note**
>
## 📝 À faire

Crée le fichier `.github/workflows/ci.yml`. Il doit :

1. **À chaque push sur `main`** : lancer les tests, puis déployer
2. **À chaque PR** : juste les tests
3. Utiliser un **secret** `HF_TOKEN` pour pousser sur HF

> 💡 **Tu bloques ?** Consulte la correction sur le site web du cours en dépliant le bloc « Voir la correction ».

## 🔑 Étape 7.2 — Configurer les secrets GitHub

> **ℹ️ Note**
>
## 📝 À faire

Sur ton repo GitHub :

1. **Settings** → **Secrets and variables** → **Actions**
2. **New repository secret** pour chacun :
   - `GROQ_API_KEY` : ta clé Groq
   - `HF_TOKEN` : ton token Hugging Face (rôle Write)
   - `HF_USERNAME` : ton username HF


## 🛡️ Étape 7.3 — Branch protection

> **ℹ️ Note**
>
## 📝 À faire

Pour éviter de pousser du code cassé sur `main` :

1. **Settings** → **Branches**
2. **Add rule** sur `main`
3. Active :
   - ✅ Require status checks to pass before merging
   - Sélectionne le job `test`


## 🎯 Étape 7.4 — Tester le pipeline complet

> **ℹ️ Note**
>
## 📝 À faire

1. Modifie quelque chose dans ton code (ex: change le titre dans `app.py`)
2. Commit + push
3. Va sur l'onglet **Actions** de ton repo GitHub
4. Regarde le pipeline **se dérouler**
5. À la fin, va sur ton HF Space → **la modif est en ligne** !

🎉 **Tu viens de mettre en place un vrai pipeline CI/CD professionnel.**


---

# Partie 8 — README professionnel

## 📝 Étape 8.1 — Le README parfait

> **ℹ️ Note**
>
## 📝 À faire

Un bon README suit cette structure :

1. **Titre** + badge de statut CI
2. **Description** courte (1-2 phrases)
3. **Démo** (lien HF Spaces + screenshot)
4. **Stack technique**
5. **Fonctionnalités**
6. **Installation locale**
7. **Architecture**
8. **Roadmap** (optionnel)
9. **Licence**

Écris ton README pour ce projet.

> 💡 **Tu bloques ?** Consulte la correction sur le site web du cours en dépliant le bloc « Voir la correction ».

---

# 🏁 Bilan et pour aller plus loin

## 🏆 Ce que tu as accompli

Tu viens de **réaliser un vrai déploiement professionnel** :

- ✅ **Code modulaire** (src/, tests/, config externe)
- ✅ **Interface utilisateur** Gradio
- ✅ **API** FastAPI prête
- ✅ **Tests** pytest avec golden dataset
- ✅ **Containerisation** Docker propre
- ✅ **Déploiement** HF Spaces (URL publique)
- ✅ **CI/CD** GitHub Actions complet
- ✅ **Monitoring** loguru structuré
- ✅ **README** professionnel
- ✅ **Secrets** sécurisés

**Tu as une URL publique.** Tu peux la mettre sur ton CV, sur LinkedIn, dans tes candidatures.

## 💼 Pour ton CV / portfolio

Sur LinkedIn ou ton CV, ajoute :

> **TechVolt Assistant — Chatbot RAG/Agent en production**
> 
> Chatbot d'assistance technique combinant RAG (26 docs) et agent function-calling (5 outils métier).
> 
> **Stack** : Python · Llama 3.3 (Groq) · ChromaDB · sentence-transformers · FastAPI · Gradio · Docker · GitHub Actions · Hugging Face Spaces
> 
> **Réalisations** : pipeline CI/CD complet, monitoring structuré, tests automatisés (golden dataset, comportement agent), déploiement reproductible.
> 
> 🎬 [Démo en ligne](URL_HF_SPACES) · 💻 [Code](URL_GITHUB)

## 🚀 Pistes d'amélioration

Pour aller **encore plus loin** sur ce projet :

1. **Évaluation continue** : un workflow nightly qui teste 50 questions et calcule des métriques (vu en 8.5)
2. **Cache des embeddings** : sauvegarder ChromaDB pour éviter le ré-encoding au démarrage
3. **Métriques Prometheus** : exposer `/metrics` pour Grafana (vu en 8.5)
4. **Authentification** : ajouter une API key pour l'API FastAPI
5. **Multi-modèles** : permettre à l'utilisateur de choisir entre Llama, Mistral, etc.
6. **A/B testing** : 2 variantes de prompts, comparer les retours utilisateurs

## 🎓 Module 8 : terminé !

Tu maîtrises maintenant **toutes les compétences MLOps** d'un junior+ en 2026 :

- ✅ Concepts MLOps (8.1)
- ✅ FastAPI (8.2)
- ✅ Docker (8.3)
- ✅ MLflow (8.4)
- ✅ Monitoring (8.5)
- ✅ CI/CD (8.6)
- ✅ **Projet vitrine déployé** (ce TP)

## 🎉 PARCOURS COMPLET TERMINÉ

Tu viens de finir **8 modules complets** :

1. ✅ **Maths pour la data** (M1)
2. ✅ **NumPy + Pandas** (M2)
3. ✅ **EDA et visualisation** (M3)
4. ✅ **ML supervisé** (M4)
5. ✅ **ML non supervisé** (M5)
6. ✅ **Deep Learning** (M6)
7. ✅ **LLM et IA générative** (M7)
8. ✅ **MLOps et déploiement** (M8)

**Tu es maintenant un·e IA Engineer.** 🎓

> **💡 Astuce**
>
## 🎯 Le défi du défi

Pour vraiment **clôturer** ce parcours :

1. Mets le **projet TechVolt** en avant dans ton portfolio GitHub
2. Écris un **post LinkedIn** : "Je viens de terminer un projet RAG/Agent déployé en prod, voici ce que j'ai appris..."
3. Postule à **3 offres** d'IA Engineer en mentionnant ce projet
4. **Adapte le projet** à un domaine qui te passionne (ta boîte, un wiki, une cause...)

**Les recruteurs ne demandent pas un master, ils demandent des projets concrets.** Tu en as un. 🚀

**Bonne continuation, ingénieur·e IA.** 💙